# Step-by-step inspection for `CascadingMitigationEnv`

这个 notebook 用于**小网络分步检测**新的 mitigation environment 是否符合设计：

- 初始失效是否先由环境触发；
- action mask 是否合理；
- `protect node` 是否生效；
- `disconnect edge` 是否会改变拓扑；
- `do nothing` 是否正常；
- 每一步后网络形态、失败节点、保护节点、断开边、负载比例是否符合预期。

建议把这个 notebook 放到：

```text
cascading_mitigation_project/notebooks/inspect_mitigation_env_step_by_step.ipynb
```

然后从项目根目录启动：

```powershell
cd cascading_mitigation_project
jupyter notebook notebooks/inspect_mitigation_env_step_by_step.ipynb
```

In [ ]:
# Cell 1: imports and path setup

from pathlib import Path
import sys
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Make imports work whether the notebook is opened from project root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from envs.cascading_mitigation_env import CascadingMitigationEnv
from envs.actions import action_to_string

print("Project root:", PROJECT_ROOT)

In [ ]:
# Cell 2: create a small environment

env = CascadingMitigationEnv(
    N=10,
    network_type="BA",
    m=2,
    alpha=0.15,
    max_steps=6,
    budget=3.0,
    protect_cost=1.0,
    disconnect_cost=1.0,
    protect_strength=0.8,
    protect_duration=2,
    failure_model="deterministic",
    redistribution_mode="uniform",
    load_type="degree",
    initial_failures=1,
    initial_failure_strategy="highest_load",
    allow_disconnect=True,
    seed=7,
)

obs, info = env.reset()

print("Initial info:")
for k, v in info.items():
    print(f"{k}: {v}")

print("\nAction dimension:", env.action_dim)
print("Do-nothing action id:", env.do_nothing_action_id)
print("Valid action ids:", np.where(obs["action_mask"] == 1)[0])

In [ ]:
# Cell 3: helper functions for inspection and visualization

def summarize_obs(obs, info=None):
    """Print compact state information."""
    failed = np.where(obs["failed_mask"] == 1)[0]
    protected = np.where(obs["protected_mask"] == 1)[0]
    valid_actions = np.where(obs["action_mask"] == 1)[0]

    print("Failed nodes:", failed.tolist())
    print("Protected nodes:", protected.tolist())
    print("Budget left:", float(obs["global_features"][1] * env.initial_budget))
    print("Valid actions:", valid_actions.tolist())

    print("\nNode table:")
    print("node | load | capacity | load/cap | failed | protected")
    for i in range(env.N):
        print(
            f"{i:>4} | "
            f"{obs['node_load'][i]:>6.3f} | "
            f"{obs['node_capacity'][i]:>8.3f} | "
            f"{obs['load_ratio'][i]:>8.3f} | "
            f"{int(obs['failed_mask'][i]):>6} | "
            f"{int(obs['protected_mask'][i]):>9}"
        )

    if info is not None:
        print("\nInfo:")
        for k, v in info.items():
            print(f"{k}: {v}")


def get_node_style(obs):
    labels = {}
    node_colors = []
    border_colors = []
    node_sizes = []

    for i in range(env.N):
        load_ratio = float(obs["load_ratio"][i])
        labels[i] = f"{i}\n{load_ratio:.2f}"

        if obs["failed_mask"][i] == 1:
            node_colors.append("lightgray")
            border_colors.append("black")
            node_sizes.append(900)
        elif obs["protected_mask"][i] == 1:
            node_colors.append("lightgreen")
            border_colors.append("black")
            node_sizes.append(950)
        else:
            node_colors.append("lightskyblue")
            border_colors.append("black")
            node_sizes.append(850)

    return labels, node_colors, border_colors, node_sizes


# Fixed layout based on the original graph, so topology changes are visually comparable across steps.
BASE_GRAPH = nx.from_numpy_array(env.base_adj)
POS = nx.spring_layout(BASE_GRAPH, seed=123)


def visualize_env(obs, title="Network state"):
    """
    Node label:
        node_id
        load/capacity ratio

    Colors:
        blue  = active normal node
        green = protected node
        gray  = failed node

    Edges:
        solid edge       = active edge
        dashed faint edge = inactive edge from original graph
    """
    active_graph = nx.from_numpy_array(obs["adj_matrix"])
    original_graph = nx.from_numpy_array(env.base_adj)

    labels, node_colors, border_colors, node_sizes = get_node_style(obs)

    plt.figure(figsize=(8, 6))
    plt.title(title)

    inactive_edges = []
    active_edges = set(tuple(sorted(e)) for e in active_graph.edges())
    for e in original_graph.edges():
        e_sorted = tuple(sorted(e))
        if e_sorted not in active_edges:
            inactive_edges.append(e)

    nx.draw_networkx_edges(
        original_graph,
        POS,
        edgelist=inactive_edges,
        style="dashed",
        alpha=0.25,
        width=1.0,
    )

    nx.draw_networkx_edges(
        active_graph,
        POS,
        edgelist=list(active_graph.edges()),
        width=2.0,
    )

    nx.draw_networkx_nodes(
        original_graph,
        POS,
        node_color=node_colors,
        edgecolors=border_colors,
        node_size=node_sizes,
        linewidths=1.5,
    )

    nx.draw_networkx_labels(
        original_graph,
        POS,
        labels=labels,
        font_size=9,
    )

    plt.axis("off")
    plt.show()


summarize_obs(obs, info)
visualize_env(obs, title="Initial state after environment-triggered initial failure")

In [ ]:
# Cell 4: list all valid actions in readable form

valid_actions = np.where(obs["action_mask"] == 1)[0]

print("Readable valid actions:")
for a in valid_actions:
    print(f"{a:>3}: {action_to_string(int(a), env.N, env.edge_list)}")

In [ ]:
# Cell 5: Step 1 — choose a protection action manually

# Strategy for inspection:
# protect the valid active node with the highest load/capacity ratio.
valid_actions = np.where(obs["action_mask"] == 1)[0]
valid_node_actions = [a for a in valid_actions if a < env.N]

if len(valid_node_actions) > 0:
    load_ratio = obs["load_ratio"]
    action = int(max(valid_node_actions, key=lambda a: load_ratio[a]))
else:
    action = env.do_nothing_action_id

print("Chosen action:", action, action_to_string(action, env.N, env.edge_list))

obs, reward, terminated, truncated, info = env.step(action)

print("\nReward:", reward)
print("Terminated:", terminated, "Truncated:", truncated)
summarize_obs(obs, info)
visualize_env(obs, title=f"After Step 1: {action_to_string(action, env.N, env.edge_list)}")

In [ ]:
# Cell 6: Step 2 — choose a valid edge disconnection action

def choose_first_valid_disconnect_action(obs):
    valid_actions = np.where(obs["action_mask"] == 1)[0]
    edge_actions = [a for a in valid_actions if env.N <= a < env.N + env.num_edges]
    if len(edge_actions) == 0:
        return env.do_nothing_action_id
    return int(edge_actions[0])


if not (terminated or truncated):
    action = choose_first_valid_disconnect_action(obs)
    print("Chosen action:", action, action_to_string(action, env.N, env.edge_list))

    obs, reward, terminated, truncated, info = env.step(action)

    print("\nReward:", reward)
    print("Terminated:", terminated, "Truncated:", truncated)
    summarize_obs(obs, info)
    visualize_env(obs, title=f"After Step 2: {action_to_string(action, env.N, env.edge_list)}")
else:
    print("Episode already ended before Step 2.")

In [ ]:
# Cell 7: Step 3 — choose do-nothing

if not (terminated or truncated):
    action = env.do_nothing_action_id
    print("Chosen action:", action, action_to_string(action, env.N, env.edge_list))

    obs, reward, terminated, truncated, info = env.step(action)

    print("\nReward:", reward)
    print("Terminated:", terminated, "Truncated:", truncated)
    summarize_obs(obs, info)
    visualize_env(obs, title="After Step 3: do nothing")
else:
    print("Episode already ended before Step 3.")

In [ ]:
# Cell 8: Continue with random valid actions until the episode ends

history = []

while not (terminated or truncated):
    valid_actions = np.where(obs["action_mask"] == 1)[0]
    action = int(np.random.choice(valid_actions))

    obs, reward, terminated, truncated, info = env.step(action)

    record = {
        "step": info["step"],
        "action": action_to_string(action, env.N, env.edge_list),
        "reward": reward,
        "damage": info["damage"],
        "failed_fraction": info["failed_fraction"],
        "lcc_ratio": info["lcc_ratio"],
        "lost_load_ratio": info["lost_load_ratio"],
        "budget_left": info["budget_left"],
    }
    history.append(record)

    print("\n" + "=" * 80)
    print("Step:", record["step"])
    print("Action:", record["action"])
    print("Reward:", record["reward"])
    print("Damage:", record["damage"])
    print("Failed fraction:", record["failed_fraction"])
    print("LCC ratio:", record["lcc_ratio"])
    print("Lost load ratio:", record["lost_load_ratio"])
    print("Budget left:", record["budget_left"])

    visualize_env(obs, title=f"After Step {record['step']}: {record['action']}")

print("\nEpisode ended.")
print("Final info:", info)

In [ ]:
# Cell 9: Plot metric evolution from the automatic continuation section

if len(history) > 0:
    steps = [r["step"] for r in history]
    damage = [r["damage"] for r in history]
    failed_fraction = [r["failed_fraction"] for r in history]
    lcc_ratio = [r["lcc_ratio"] for r in history]
    lost_load_ratio = [r["lost_load_ratio"] for r in history]

    plt.figure(figsize=(7, 4))
    plt.plot(steps, damage, marker="o", label="damage")
    plt.plot(steps, failed_fraction, marker="o", label="failed_fraction")
    plt.plot(steps, lcc_ratio, marker="o", label="lcc_ratio")
    plt.plot(steps, lost_load_ratio, marker="o", label="lost_load_ratio")
    plt.xlabel("Step")
    plt.ylabel("Metric value")
    plt.title("Cascade mitigation metrics over steps")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No automatic continuation history to plot.")

## 如何判断环境设计是否准确

运行 notebook 后重点检查：

1. **reset 后是否已经有 initial failed node**  
   这说明 agent 不是攻击者，而是在初始故障后做 mitigation。

2. **failed node 是否从 active topology 中移除**  
   灰色节点应当没有有效连接，或其原始边显示为 inactive edge。

3. **protect node 是否改变 protected status**  
   保护节点应显示为 protected，并在接下来若干步内具有更高 effective capacity。

4. **disconnect edge 是否只移除边，不直接删除节点**  
   如果断边后节点仍存在，说明 action 语义正确。

5. **do-nothing 是否不改变结构但仍推进 cascade**  
   如果存在 overload，do-nothing 后仍可能发生新的失效。

6. **reward 是否符合 mitigation 方向**  
   当 action 使 damage 下降或减少传播时，reward 应更高；当 damage 增加或 action cost 较高时，reward 应降低。